<div dir="rtl" align="right">

# تجزیه مقدارهای تکین (SVD) و فشرده‌سازی

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## چرا SVD؟
بردار ویژه فقط برای ماتریس مربعی تعریف می‌شود، ولی داده‌های ما معمولاً مستطیلی‌اند ($n$ نمونه × $d$ ویژگی). SVD برای **هر** ماتریسی کار می‌کند:

$$A=U\Sigma V^\top$$

| بخش | شکل | چیست |
|---|---|---|
| $U$ | $m\times m$ | ستون‌ها متعامد؛ «الگوهای سطرها» |
| $\Sigma$ | $m\times n$ قطری | مقدارهای تکین $\sigma_1\ge\sigma_2\ge\dots\ge0$؛ اهمیت هر الگو |
| $V^\top$ | $n\times n$ | سطرها متعامد؛ «الگوهای ستون‌ها» |

از نظر هندسی هر ماتریس = **چرخش** ($V^\top$) + **کشیدن در راستای محورها** ($\Sigma$) + **چرخش دوباره** ($U$).

## مثال عددی
برای $A=\begin{bmatrix}3&2&2\\2&3&-2\end{bmatrix}$:

| گام | محاسبه |
|---|---|
| 1. $AA^\top$ | $\begin{bmatrix}17&8\\8&17\end{bmatrix}$ |
| 2. مقدارهای ویژه $AA^\top$ | $(17-\lambda)^2-64=0\Rightarrow\lambda=25,\ 9$ |
| 3. مقدارهای تکین | $\sigma_1=\sqrt{25}=5,\quad\sigma_2=\sqrt9=3$ |
| 4. ستون‌های U | $\frac{1}{\sqrt2}(1,1)$ و $\frac{1}{\sqrt2}(1,-1)$ |

پس مقدارهای تکین، **جذر مقدارهای ویژه $A^\top A$** هستند.

## تقریب با رتبه پایین
SVD ماتریس را به جمع چند «لایه» ساده می‌شکند:

$$A=\sigma_1 u_1 v_1^\top+\sigma_2 u_2 v_2^\top+\dots$$

اگر فقط $k$ جمله اول را نگه داریم، **بهترین تقریب ممکن با رتبه k** را داریم (قضیه اکارت–یانگ). چون مقدارهای تکین به‌سرعت کوچک می‌شوند، چند جمله اول بیشتر اطلاعات را دارند.

> [یک عکس سیاه‌وسفید که با k مقدار تکین اول بازسازی شده است. با 20 مقدار، تصویر قابل تشخیص است.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_svd_img.png)

*یک عکس سیاه‌وسفید که با k مقدار تکین اول بازسازی شده است. با 20 مقدار، تصویر قابل تشخیص است.*

> [چپ: مقدارهای تکین خیلی سریع کوچک می‌شوند. راست: سهم تجمعی «انرژی».](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_svd_energy.png)

*چپ: مقدارهای تکین خیلی سریع کوچک می‌شوند. راست: سهم تجمعی «انرژی».*

## چقدر صرفه‌جویی؟
یک تصویر $m\times n$ در حالت عادی $mn$ عدد است. با رتبه k فقط $k(m+n+1)$ عدد لازم است:

| تصویر $200\times300$ | تعداد عدد | درصد |
|---|---|---|
| اصلی | 60,000 | 100٪ |
| k = 50 | $50\times501=25{,}050$ | 42٪ |
| k = 20 | $20\times501=10{,}020$ | 17٪ |
| k = 5 | $5\times501=2{,}505$ | 4٪ |

## کاربردها
| کاربرد | چطور |
|---|---|
| PCA | SVD روی داده مرکزی‌شده؛ ستون‌های V مؤلفه‌های اصلی‌اند و $\sigma_i^2/(n-1)$ واریانس هر کدام |
| سیستم توصیه‌گر | ماتریس کاربر × فیلم را با رتبه پایین تقریب می‌زنیم تا امتیازهای خالی پیش‌بینی شوند |
| تحلیل معنایی متن (LSA) | SVD روی ماتریس سند × واژه، «موضوع‌ها» را پیدا می‌کند |
| شبه‌وارون | $A^+=V\Sigma^+U^\top$؛ حل کمترین مربعات حتی وقتی $X^\top X$ تکین است |
| LoRA در مدل‌های زبانی | تغییر وزن‌ها را با دو ماتریس کوچک رتبه پایین نشان می‌دهند |



</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[3.0, 2.0, 2.0],
              [2.0, 3.0, -2.0]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("singular values:", s)                       # [5. 3.]
print("U ="); print(U.round(3))
print("rebuilt A ="); print(((U * s) @ Vt).round(6))
print("sqrt of eigenvalues of A A^T:", np.sqrt(np.linalg.eigvalsh(A @ A.T))[::-1])

# Rank-1 approximation: keep only the largest singular value
A1 = s[0] * np.outer(U[:, 0], Vt[0])
print("rank-1 approximation ="); print(A1.round(3))
print("error (Frobenius norm) =", round(np.linalg.norm(A - A1), 3), " = second singular value")

# Compress a synthetic 120 x 160 "image": smooth pattern + a bright square + a little noise
rng = np.random.default_rng(0)
yy, xx = np.mgrid[0:120, 0:160]
img = np.sin(xx / 14) * np.cos(yy / 18) + 0.05 * rng.normal(size=(120, 160))
img[40:80, 60:110] += 1.5
U, s, Vt = np.linalg.svd(img, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, k in zip(axes, [1, 3, 10, 120]):
    approx = (U[:, :k] * s[:k]) @ Vt[:k]
    err = np.linalg.norm(img - approx) / np.linalg.norm(img)
    ax.imshow(approx, cmap="gray"); ax.axis("off")
    ax.set_title(f"k = {k}, error {100 * err:.1f}%")
plt.tight_layout(); plt.show()
print("energy kept by the first 10:", round(np.sum(s[:10] ** 2) / np.sum(s ** 2), 4))

<div dir="rtl" align="right">

### تمرین

ماتریس امتیاز 4 کاربر به 3 فیلم $\begin{bmatrix}5&4&1\\4&5&1\\1&1&5\\2&1&4\end{bmatrix}$ را با رتبه 2 تقریب بزن. خطای نسبی چقدر است؟

</div>

In [ ]:
# Your answer here